# COMP64702 RAG Coursework  
## Notebook 1: Corpus and Benchmark  

**Chosen cuisine:** South Asian cuisine  

---

### This notebook:
- loads and validates the background corpus  
- inspects the manually curated benchmark dataset  
- checks alignment between benchmark and corpus  
- previews the input query file used in the RAG pipeline  

In [14]:
import json
from pathlib import Path
import pandas as pd

In [15]:
DATA_DIR = Path(".")

corpus_path = DATA_DIR / "background_corpus.json"
benchmark_path = DATA_DIR / "benchmark_dataset.json"
benchmark_input_path = DATA_DIR / "input_payload.json"

In [16]:
with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} corpus documents")

Loaded 204 corpus documents


In [17]:
required_fields = ["doc_id", "title"]

missing_field_count = 0

for i, doc in enumerate(corpus):
    for field in required_fields:
        if field not in doc:
            print(f"Warning: Document {i} missing field '{field}'")
            missing_field_count += 1

print("Corpus validation complete")
print(f"Total missing required fields found: {missing_field_count}")

Corpus validation complete
Total missing required fields found: 0


In [18]:
num_docs = len(corpus)
num_with_sections = sum(1 for d in corpus if d.get("sections"))
num_with_full_text = sum(1 for d in corpus if d.get("full_text", "").strip())

total_chars = sum(len(d.get("full_text", "")) for d in corpus)
avg_chars = total_chars / num_docs if num_docs > 0 else 0

total_sections = sum(len(d.get("sections", [])) for d in corpus)
avg_sections = total_sections / num_docs if num_docs > 0 else 0

print("Corpus Summary:")
print(f"- Documents: {num_docs}")
print(f"- Documents with full_text: {num_with_full_text}")
print(f"- Documents with sections: {num_with_sections}")
print(f"- Total sections: {total_sections}")
print(f"- Avg sections per document: {avg_sections:.2f}")
print(f"- Avg document length: {avg_chars:.0f} characters")

Corpus Summary:
- Documents: 204
- Documents with full_text: 204
- Documents with sections: 21
- Total sections: 142
- Avg sections per document: 0.70
- Avg document length: 1707 characters


In [19]:
source_counts = pd.Series([doc.get("source", "unknown") for doc in corpus]).value_counts()

print("Source distribution:")
print(source_counts)

Source distribution:
unknown    204
Name: count, dtype: int64


In [20]:
example_doc = corpus[0]

print("Example Document")
print("TITLE :", example_doc.get("title", "N/A"))
print("DOC ID:", example_doc.get("doc_id", "N/A"))
print("SOURCE:", example_doc.get("source", "N/A"))
print()

if example_doc.get("sections"):
    print("SECTIONS:")
    for sec in example_doc["sections"][:3]:
        print("-", sec.get("heading", "Untitled section"))
else:
    print("TEXT PREVIEW:\n")
    print(example_doc.get("full_text", "")[:800])

Example Document
TITLE : South Asian corpus scope and allowed sources
DOC ID: corpus_scope_and_sources_note
SOURCE: N/A

TEXT PREVIEW:

This corpus file is structured for a South Asian cuisine RAG project. It is designed to include all countries requested by the user and to go one level deeper into regional or state cuisines where appropriate. Only the publicly listed sources in the coursework brief should be used when replacing scaffold entries with full article text: Wikipedia, Wikibooks, and Around the World in 80 Cuisines.


In [21]:
with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

print(f"Loaded {len(benchmark)} benchmark questions")

Loaded 100 benchmark questions


In [22]:
required_benchmark_fields = [
    "query_id",
    "query",
    "answer",
    "answer_source",
    "question_type",
    "difficulty"
]

missing_benchmark_fields = 0

for i, item in enumerate(benchmark):
    for field in required_benchmark_fields:
        if field not in item:
            print(f"Warning: Benchmark item {i} missing field '{field}'")
            missing_benchmark_fields += 1

print("Benchmark validation complete")
print(f"Total missing required benchmark fields found: {missing_benchmark_fields}")

Benchmark validation complete
Total missing required benchmark fields found: 0


In [23]:
benchmark_df = pd.DataFrame(benchmark)

display_cols = ["query_id", "query", "answer_source", "question_type", "difficulty"]
print(benchmark_df[display_cols].head(10))

  query_id                                              query  \
0   sa_000  What is biryani and which culinary tradition i...   
1   sa_001  What is butter chicken and where did it origin...   
2   sa_002                 What is paneer and how is it made?   
3   sa_003  What is ghee and why is it important in Indian...   
4   sa_004  What is dal makhani and what makes it distinct...   
5   sa_005  What is garam masala and how does it differ fr...   
6   sa_006  What is the dum cooking technique and which di...   
7   sa_007  What is tadka and when is it used in Indian co...   
8   sa_008  What is tamarind and how is it used in Indian ...   
9   sa_009  What are the key characteristics of Rajasthani...   

                       answer_source question_type difficulty  
0                 india_dish_biryani       factual       easy  
1          india_dish_butter_chicken       factual       easy  
2            india_ingredient_paneer       factual       easy  
3              india_ingredi

In [24]:
print("Benchmark Summary:")
print(f"- Total questions: {len(benchmark_df)}")

print("\nQuestion type distribution:")
print(benchmark_df["question_type"].value_counts())

print("\nDifficulty distribution:")
print(benchmark_df["difficulty"].value_counts())

Benchmark Summary:
- Total questions: 100

Question type distribution:
question_type
factual        48
procedural     18
comparative    17
ingredient     12
technique       5
Name: count, dtype: int64

Difficulty distribution:
difficulty
medium    55
hard      24
easy      21
Name: count, dtype: int64


In [25]:
corpus_ids = set(doc.get("doc_id") for doc in corpus)

missing_sources = []
for item in benchmark:
    if item["answer_source"] not in corpus_ids:
        missing_sources.append(item["answer_source"])

if missing_sources:
    print("Missing benchmark answer sources in corpus:")
    print(sorted(set(missing_sources)))
else:
    print("All benchmark answer sources exist in the corpus")

Missing benchmark answer sources in corpus:
['india_technique_tadka']


In [26]:
with open(benchmark_input_path, "r", encoding="utf-8") as f:
    benchmark_input_payload = json.load(f)

print("Preview of benchmark_input_only.json:")
print(benchmark_input_payload["queries"][:5])

Preview of benchmark_input_only.json:
[{'query_id': 'sa_000', 'query': 'What is biryani and which culinary tradition is it most closely associated with?'}, {'query_id': 'sa_001', 'query': 'What is butter chicken and where did it originate?'}, {'query_id': 'sa_002', 'query': 'What is paneer and how is it made?'}, {'query_id': 'sa_003', 'query': 'What is ghee and why is it important in Indian cooking?'}, {'query_id': 'sa_004', 'query': 'What is dal makhani and what makes it distinctive?'}]


In [27]:
print("State Summary:")
print(f"- Corpus documents: {len(corpus)}")
print(f"- Benchmark questions: {len(benchmark)}")
print(f"- Input-only queries loaded: {len(benchmark_input_payload['queries'])}")
print("- Corpus and benchmark successfully validated")

State Summary:
- Corpus documents: 204
- Benchmark questions: 100
- Input-only queries loaded: 100
- Corpus and benchmark successfully validated
